# Testing the MLIP on graphite

In [ ]:
# Not relevant during the school
import jax
import numpy as np
import matplotlib.pyplot as plt
jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_enable_x64", True)


# Importing ASE modules
import ase
from ase.build import bulk, sort
from ase.optimize import BFGS
from ase.build.supercells import make_supercell
from ase.visualize import view
from ase.io.vasp import read_vasp
from ase.filters import StrainFilter # FrechetCellFilter, 


# Importing our calculator
from panna.interfaces.jax_ASE import PANNAJAXCalculator
from panna.interfaces.phonon_utils import get_phonons, plot_phonon_mode

In [ ]:
# =============================================================================
# create unit cell and supercell
#uc = bulk('C','diamond', a=3.54, cubic=False) 
uc = read_vasp('div_app/POSCARS/POSCAR_graphite')
natuc = uc.get_global_number_of_atoms() 
N1, N2, N3 = 4,4,4
sc = make_supercell(uc, [[N1, 0, 0], [0, N2, 0], [0, 0, N3]])
R0 = sc.get_positions()
SCell = np.array(sc.get_cell())
natsc = sc.get_global_number_of_atoms()
masses = uc.get_masses() 
# =============================================================================

## Graphite structure
Inspect the supercell. Comment on the difference with the diamond structure

In [ ]:
view(sc)

## Computing phonons

Assignment: compute the phonons in graphite

In [ ]:
# =============================================================================
# import PANNA weights
configfile = 'div_app/train_diverse/mytrain_diverse.ini' # 'dia_100_app/train_diamond_small/train.ini' #
pcalc = PANNAJAXCalculator(config=configfile) #, weights_file='epoch_930_step_93000.pkl') 
sc.calc = pcalc
# =============================================================================

In [ ]:
# =============================================================================
# relax the structure, with standard BFGS (no cell relaxation)

# check the that the forces are small enough
# =============================================================================

Consider the BZ of graphite. Why is it squashed in the z-axis?

In [ ]:
%matplotlib widget
path = sc.cell.bandpath("GMKGA", npoints=5)
path.plot(show=True)
print(path.kpts)

In [ ]:
# =============================================================================
# Phonons
# fill in the kpoints with the correct numbers. Hint: use the points above!
kinput_scaled = np.array([
    [],   # Γ
    [],   # M
    [],   # K
    [],   # Γ
    []    # A
])

labels = [] # fill the labels with their names!
interp = 10
tol = 1e-5
asr = True
dx_and_sx = True
save_freqs_eigvecs = False
connect_bands = True
verbose = True
appendix = ''
inputs_phonons = (natuc, N1, N2, N3, kinput_scaled, labels, interp, masses)
x, xcom, xlabels, kps_sc, kps, frequencies_list, eigvecs_list, Ds_list, K = get_phonons(sc, inputs_phonons, asr=asr, dx_and_sx=dx_and_sx, save_freqs_eigvecs=save_freqs_eigvecs, appendix=appendix, tol=tol, connect_bands=connect_bands, verbose=verbose)
# =============================================================================

In [ ]:
# =============================================================================
# band plotting
# plot the phonon frequencies, contained in frequencies_list, on the path, contained in x
ax.set_ylabel('THz'), ax.set_title('Phonons in graphite')
# =============================================================================

Plot the acoustic phonon modes near $\Gamma$. Comment on the dispersion. 
1) Do you notice any difference between in-plane and out of plane modes?
2) The material is softer with respect to a in-plane stretching or an out of plane stretching?

In [ ]:
%matplotlib widget
ik, n = 56, 1 # second k-point, third mode
kpt = kps[ik]
omega = frequencies_list[n,ik]
eigvec = eigvecs_list[ik,:,n]
plot_phonon_mode(
    k_cart=kpt,
    n=n,
    omega=omega,
    eigvec=eigvec,
    Ruc=uc.get_positions(),
    Cell=uc.get_cell().array,
    masses=uc.get_masses(),
    symbols=uc.get_chemical_symbols(),
    ampl=10,
    repeat=(1,1,1),
    k_scale=0.5,
    #show_imag=True,
)